In [1]:
"""
CLIP + FakeNewsNet — binary fake-news detection using TEXT + IMAGE + METADATA.
Metadata fields (source, authors, publish_date, subject) are prepended as a
structured prefix to the article text before CLIP tokenisation.

Model  : openai/clip-vit-base-patch32
         Text encoder  (last 2 blocks unfrozen) → 512-d projection
         Vision encoder (last 2 blocks unfrozen) → 512-d projection
         Fusion: cat(512, 512) → LayerNorm → Linear(512) → GELU
                 → Dropout(0.3) → Linear(2)

Training design
---------------
  • Partial CLIP freeze — only the last `unfreeze_last_n` transformer
    blocks + projection layers are trained; the rest stays frozen.
  • Differential LR — backbone at lr × 0.1, classifier head at lr.
  • Gradient clipping (norm ≤ 1.0).
  • Label smoothing (0.1).
  • Overfitting tracking — train/val/test loss, accuracy and F1 reported
    every epoch; gap (Train − Val) and severity label shown inline.

Labels : 0 = real,  1 = fake
Dataset : rickstello/FakeNewsNet  (HuggingFace)
Split   : 80 % train pool / 10 % val / 10 % test  (stratified)
CV      : 5-fold StratifiedKFold on the 80 % train pool
Epochs  : 5 per fold
Metrics : Accuracy, Precision, Recall, F1  (binary, pos_label=1=fake)
Output  : printed ASCII table + two CSV files

Run: python clip_fakenewsnet_kaggle_80_10_10_ep5_with_metadata.py
"""

from __future__ import annotations

import csv
import io
import os
import random
from collections.abc import Callable
from contextlib import nullcontext
from dataclasses import dataclass
from typing import Any

import numpy as np
import requests
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor, get_linear_schedule_with_warmup

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ── Kaggle / HF offline guard ─────────────────────────────────────────────────
# If HF_DATASETS_OFFLINE is already set (Kaggle no-internet kernel), respect it.
# Otherwise leave it unset so the Hub attempt in load_hf_dataset() can proceed.
_HF_OFFLINE = os.environ.get("HF_DATASETS_OFFLINE", "0") == "1"
if _HF_OFFLINE:
    print("HF_DATASETS_OFFLINE=1 detected — will skip HuggingFace Hub and use kagglehub.")


# ── Configuration ─────────────────────────────────────────────────────────────
@dataclass
class Config:
    model_name: str = "openai/clip-vit-base-patch32"
    dataset_name: str = "rickstello/FakeNewsNet"
    dataset_split: str = "train"
    # Kaggle dataset slug for offline loading (used when HF Hub is unreachable)
    kaggle_dataset_slug: str = "rickstello/fakenewsnet"  # kagglehub owner/dataset-slug
    max_samples: int = 10000
    num_classes: int = 2
    batch_size: int = 16
    epochs: int = 5
    lr: float = 2e-5
    backbone_lr_mult: float = 0.1     # CLIP backbone lr = lr × this
    weight_decay: float = 0.01
    clip_grad_norm: float = 1.0       # gradient clipping max norm
    label_smoothing: float = 0.1      # prevents overconfident predictions
    dropout: float = 0.3
    grad_accumulation: int = 1
    unfreeze_last_n: int = 2          # CLIP transformer blocks to unfreeze
    n_splits: int = 5
    num_workers: int = 0
    image_timeout_s: float = 10.0
    image_size: int = 224             # CLIP ViT-B/32 native resolution
    text_max_length: int = 77         # CLIP tokenizer context window
    train_frac: float = 0.8
    val_frac: float = 0.1
    test_frac: float = 0.1
    results_dir: str = "."
    results_file_prefix: str = "clip_fakenewsnet_with_metadata"
    # Metadata fields to include as a text prefix (set to [] to disable)
    metadata_fields: tuple = ("source", "authors", "publish_date", "subject")
    metadata_max_chars: int = 120   # max chars per metadata field value
    force_cpu: bool = False
    require_image_source: bool = False  # True → skip rows with no image


config = Config()


# ── Device / AMP helpers ──────────────────────────────────────────────────────
def resolve_training_device(*, force_cpu: bool = False) -> torch.device:
    if force_cpu:
        print("force_cpu=True: using CPU.")
        return torch.device("cpu")
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        major, _ = torch.cuda.get_device_capability(0)
        name = torch.cuda.get_device_name(0)
    except Exception as exc:
        print(f"Could not query CUDA device ({exc}); using CPU.")
        return torch.device("cpu")
    if major < 7:
        print(
            f"WARNING: GPU '{name}' is compute {major}.x; "
            "using CPU (PyTorch wheels require sm_70+). Use T4+ on Kaggle."
        )
        return torch.device("cpu")
    try:
        t = torch.zeros(1, device="cuda")
        _ = (t + 1.0).item()
        del t
        torch.cuda.synchronize()
    except Exception as exc:
        print(f"WARNING: CUDA probe failed ({type(exc).__name__}: {exc}); using CPU.")
        return torch.device("cpu")
    return torch.device("cuda")


def build_amp_helpers(use_cuda: bool) -> tuple[Callable[[], Any], Any]:
    if not use_cuda:
        return (lambda: nullcontext()), torch.cuda.amp.GradScaler(enabled=False)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        return (lambda: torch.amp.autocast("cuda")), scaler
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=True)
        return (lambda: torch.cuda.amp.autocast()), scaler


# ── Image / text helpers ──────────────────────────────────────────────────────
def load_pil_from_url(url: str, timeout: float) -> Image.Image | None:
    try:
        r = requests.get(url, timeout=timeout, stream=True)
        r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    except Exception:
        return None


def get_article_text(ex: dict) -> str:
    """Return raw article text (no metadata). Kept for label/index scanning."""
    for key in ("title", "text", "article", "headline", "content", "body"):
        v = ex.get(key)
        if v is not None and str(v).strip() and str(v).lower() != "null":
            return str(v).strip()
    return ""


def build_text_with_metadata(ex: dict) -> str:
    """
    Build the input string: metadata prefix + article text.

    Metadata prefix format (one field per line, only present if non-empty):
        source: <value>
        authors: <value>
        publish_date: <value>
        subject: <value>
        ---
        <article text>

    Any field absent or null in the example is silently skipped.
    All values are truncated to config.metadata_max_chars characters.
    """
    parts: list[str] = []
    for field in config.metadata_fields:
        v = ex.get(field)
        if v is None:
            continue
        # Handle list values (e.g. authors stored as a list)
        if isinstance(v, (list, tuple)):
            v = ", ".join(str(x) for x in v if x)
        v = str(v).strip()
        if not v or v.lower() in ("null", "none", ""):
            continue
        v = v[: config.metadata_max_chars]
        parts.append(f"{field}: {v}")

    article_text = get_article_text(ex)
    if parts:
        prefix = "\n".join(parts) + "\n---\n"
        return prefix + article_text
    return article_text


def get_image_url(ex: dict) -> str | None:
    for key in (
        "image_url", "top_img_url", "top_img",
        "image", "img_url", "tweet_image_url", "thumbnail",
    ):
        v = ex.get(key)
        if v is None:
            continue
        if isinstance(v, str) and v.strip() and v.lower() not in ("null", "none", ""):
            return v.strip()
    return None


def decode_hf_image_field(ex: dict) -> Image.Image | None:
    v = ex.get("image")
    if v is None:
        return None
    if isinstance(v, Image.Image):
        return v.convert("RGB")
    if isinstance(v, dict):
        b = v.get("bytes")
        if b:
            try:
                return Image.open(io.BytesIO(b)).convert("RGB")
            except Exception:
                return None
    return None


def get_binary_label_fake_is_one(ex: dict) -> int | None:
    """0 = real, 1 = fake.  Derived from 'real', 'is_fake', or 'label'."""
    if ex.get("real") is not None:
        try:
            r = int(ex["real"])
            if r in (0, 1):
                return 1 - r
        except (TypeError, ValueError):
            pass
    if ex.get("is_fake") is not None:
        try:
            return int(ex["is_fake"])
        except (TypeError, ValueError):
            pass
    if ex.get("label") is not None:
        try:
            lab = int(ex["label"])
            if lab in (0, 1):
                return lab
        except (TypeError, ValueError):
            pass
    return None


def load_rgb_image_for_example(ex: dict) -> Image.Image:
    url = get_image_url(ex)
    if url:
        pil = load_pil_from_url(url, config.image_timeout_s)
        if pil is not None:
            return pil
    pil = decode_hf_image_field(ex)
    if pil is not None:
        return pil
    return Image.new("RGB", (config.image_size, config.image_size), color=(0, 0, 0))


# ── Dataset ───────────────────────────────────────────────────────────────────
class FakeNewsNetCLIPDataset(Dataset):
    """
    Loads (metadata + article text, image) pairs and
    returns CLIP-processed tensors.

    Item keys:
        input_ids       – (seq_len,)
        attention_mask  – (seq_len,)
        pixel_values    – (3, 224, 224)
        labels          – ()  long
    """

    def __init__(
        self,
        hf_dataset,
        processor: CLIPProcessor,
        indices: list[int] | None = None,
    ):
        self.rows = hf_dataset
        self.processor = processor
        self.indices = indices if indices is not None else list(range(len(hf_dataset)))

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        i = self.indices[idx]
        ex = self.rows[i]

        text = build_text_with_metadata(ex)   # metadata prefix + article text
        if not text:
            raise ValueError("empty text")

        lab = get_binary_label_fake_is_one(ex)
        if lab is None:
            raise ValueError("missing binary label")

        img = load_rgb_image_for_example(ex)
        s = config.image_size
        if img.size != (s, s):
            img = img.resize((s, s), Image.BICUBIC)

        enc = self.processor(
            text=text,
            images=img,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=config.text_max_length,
        )
        item: dict[str, torch.Tensor] = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(lab, dtype=torch.long)
        return item


def collate_fn(batch: list[dict]) -> dict[str, torch.Tensor]:
    labels = torch.stack([b.pop("labels") for b in batch], dim=0)
    keys = batch[0].keys()
    out: dict[str, torch.Tensor] = {k: torch.stack([b[k] for b in batch], dim=0) for k in keys}
    out["labels"] = labels
    return out


# ── CLIP-based binary classifier ──────────────────────────────────────────────
class ClipFakeNewsDetector(nn.Module):
    """
    Binary fake-news detector using CLIP text + image embeddings.

    Architecture
    ────────────
      text_feat   (B, 512) ──┐
                              cat → (B,1024) → LayerNorm → Linear(512)
      image_feat  (B, 512) ──┘             → GELU → Dropout → Linear(2)

    Partial CLIP freeze
    ───────────────────
      All parameters frozen except:
        • Last `unfreeze_last_n` transformer blocks in text_model.encoder
        • Last `unfreeze_last_n` transformer blocks in vision_model.encoder
        • text_projection and visual_projection  (always trainable)
    """

    def __init__(
        self,
        num_classes: int = 2,
        model_name: str = config.model_name,
        dropout: float = config.dropout,
        unfreeze_last_n: int = config.unfreeze_last_n,
    ):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)
        proj_dim: int = self.clip.config.projection_dim    # 512 for ViT-B/32

        # Freeze everything, then selectively unfreeze
        for param in self.clip.parameters():
            param.requires_grad = False

        for block in self.clip.text_model.encoder.layers[-unfreeze_last_n:]:
            for param in block.parameters():
                param.requires_grad = True

        for block in self.clip.vision_model.encoder.layers[-unfreeze_last_n:]:
            for param in block.parameters():
                param.requires_grad = True

        for param in self.clip.text_projection.parameters():
            param.requires_grad = True
        for param in self.clip.visual_projection.parameters():
            param.requires_grad = True

        fusion_dim = proj_dim * 2    # 1024
        self.head = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, num_classes),
        )

    def backbone_params(self) -> list:
        return [p for n, p in self.named_parameters()
                if p.requires_grad and n.startswith("clip.")]

    def head_params(self) -> list:
        return list(self.head.parameters())

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        pixel_values: torch.Tensor,
        labels: torch.Tensor | None = None,
        loss_fn: nn.Module | None = None,
        **unused_kwargs,
    ) -> dict[str, torch.Tensor | None]:
        clip_out = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True,
        )
        text_feat: torch.Tensor = clip_out.text_embeds    # (B, 512)
        image_feat: torch.Tensor = clip_out.image_embeds  # (B, 512)

        fused = torch.cat([text_feat, image_feat], dim=-1)  # (B, 1024)
        logits = self.head(fused)

        loss: torch.Tensor | None = None
        if labels is not None:
            criterion = loss_fn if loss_fn is not None else nn.CrossEntropyLoss()
            loss = criterion(logits, labels)
        return {"loss": loss, "logits": logits}


# ── Valid-index scan ──────────────────────────────────────────────────────────
def row_has_image_source(ex: dict) -> bool:
    return get_image_url(ex) is not None or decode_hf_image_field(ex) is not None


def build_valid_indices(hf_dataset, max_n: int) -> list[int]:
    valid: list[int] = []
    n = min(len(hf_dataset), max_n)
    for i in tqdm(range(n), desc="Scanning samples"):
        ex = hf_dataset[i]
        if not get_article_text(ex):
            continue
        if get_binary_label_fake_is_one(ex) is None:
            continue
        if config.require_image_source and not row_has_image_source(ex):
            continue
        valid.append(i)
    return valid


# ── Overfitting severity ──────────────────────────────────────────────────────
def overfitting_severity(gap: float) -> str:
    """
    Classify overfitting from the Train − Val accuracy (or F1) gap.
      gap < 0     → Under-fit
      gap < 0.02  → None
      gap < 0.05  → Mild
      gap < 0.10  → Moderate
      gap >= 0.10 → Severe
    """
    if gap < 0.0:
        return "Under-fit"
    if gap < 0.02:
        return "None"
    if gap < 0.05:
        return "Mild"
    if gap < 0.10:
        return "Moderate"
    return "Severe"


# ── Evaluation helpers ────────────────────────────────────────────────────────
def collect_predictions_with_loss(
    model: nn.Module,
    data_loader: DataLoader,
    device: torch.device,
    autocast_ctx: Callable[[], Any],
    loss_fn: nn.Module,
) -> tuple[np.ndarray, np.ndarray, float]:
    """Returns (y_true, y_pred, avg_loss)."""
    model.eval()
    all_preds, all_y = [], []
    total_loss, total_n = 0.0, 0
    with torch.no_grad():
        for batch in data_loader:
            labels = batch.pop("labels").to(device)
            n = labels.size(0)
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast_ctx():
                out = model(**batch, labels=labels, loss_fn=loss_fn)
            total_loss += out["loss"].item() * n
            total_n += n
            all_preds.append(out["logits"].argmax(dim=-1).cpu().numpy())
            all_y.append(labels.cpu().numpy())
    return np.concatenate(all_y), np.concatenate(all_preds), total_loss / max(total_n, 1)


def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
    }


# ── Overfitting report printers ───────────────────────────────────────────────
def print_fold_overfitting_table(fold_idx: int, epoch_logs: list[dict]) -> None:
    w = {"ep": 5, "loss": 8, "acc": 7, "f1": 7, "gap": 8, "sev": 10}
    hdr = (
        f"  {'Ep':>{w['ep']}}  "
        f"{'TrLoss':>{w['loss']}}  {'VaLoss':>{w['loss']}}  {'ΔLoss':>{w['gap']}}  "
        f"{'TrAcc':>{w['acc']}}  {'VaAcc':>{w['acc']}}  {'ΔAcc':>{w['gap']}}  "
        f"{'TrF1':>{w['f1']}}  {'VaF1':>{w['f1']}}  {'ΔF1':>{w['gap']}}  "
        f"{'Status':>{w['sev']}}"
    )
    sep = "  " + "-" * (len(hdr) - 2)
    print(f"\n  ── Fold {fold_idx + 1} Overfitting per Epoch ──")
    print(hdr)
    print(sep)
    for row in epoch_logs:
        print(
            f"  {row['epoch']:>{w['ep']}}  "
            f"{row['train_loss']:>{w['loss']}.4f}  {row['val_loss']:>{w['loss']}.4f}  "
            f"{row['gap_loss']:>+{w['gap']}.4f}  "
            f"{row['train_acc']:>{w['acc']}.4f}  {row['val']['accuracy']:>{w['acc']}.4f}  "
            f"{row['gap_acc']:>+{w['gap']}.4f}  "
            f"{row['train_f1']:>{w['f1']}.4f}  {row['val']['f1']:>{w['f1']}.4f}  "
            f"{row['gap_f1']:>+{w['gap']}.4f}  "
            f"{row['overfitting']:>{w['sev']}}"
        )
    print(sep)


def print_overfitting_report(fold_reports: list[dict]) -> None:
    w = {"fold": 5, "loss": 8, "acc": 7, "f1": 7, "gap": 8, "sev": 10}
    hdr = (
        f"  {'Fold':>{w['fold']}}  "
        f"{'TrLoss':>{w['loss']}}  {'VaLoss':>{w['loss']}}  {'ΔLoss':>{w['gap']}}  "
        f"{'TrAcc':>{w['acc']}}  {'VaAcc':>{w['acc']}}  {'ΔAcc':>{w['gap']}}  "
        f"{'TrF1':>{w['f1']}}  {'VaF1':>{w['f1']}}  {'ΔF1':>{w['gap']}}  "
        f"{'Status':>{w['sev']}}"
    )
    sep = "  " + "-" * (len(hdr) - 2)

    print("\n" + "=" * 96)
    print("OVERFITTING ANALYSIS — best-val-F1 epoch per fold")
    print("  Gap = Train − Val  |  positive gap → model overfits training set")
    print("=" * 96)
    print(hdr)
    print(sep)

    tr_losses, va_losses, gap_losses = [], [], []
    tr_accs,   va_accs,   gap_accs   = [], [], []
    tr_f1s,    va_f1s,    gap_f1s    = [], [], []

    for fr in fold_reports:
        b = fr["best_by_val_f1"]
        tr_losses.append(b["train_loss"]);  va_losses.append(b["val_loss"])
        tr_accs.append(b["train_acc"]);     va_accs.append(b["val"]["accuracy"])
        tr_f1s.append(b["train_f1"]);       va_f1s.append(b["val"]["f1"])
        gap_losses.append(b["gap_loss"])
        gap_accs.append(b["gap_acc"])
        gap_f1s.append(b["gap_f1"])
        print(
            f"  {fr['fold']:>{w['fold']}}  "
            f"{b['train_loss']:>{w['loss']}.4f}  {b['val_loss']:>{w['loss']}.4f}  "
            f"{b['gap_loss']:>+{w['gap']}.4f}  "
            f"{b['train_acc']:>{w['acc']}.4f}  {b['val']['accuracy']:>{w['acc']}.4f}  "
            f"{b['gap_acc']:>+{w['gap']}.4f}  "
            f"{b['train_f1']:>{w['f1']}.4f}  {b['val']['f1']:>{w['f1']}.4f}  "
            f"{b['gap_f1']:>+{w['gap']}.4f}  "
            f"{b['overfitting']:>{w['sev']}}"
        )

    print(sep)
    mean_gap_acc = float(np.mean(gap_accs))
    print(
        f"  {'MEAN':>{w['fold']}}  "
        f"{float(np.mean(tr_losses)):>{w['loss']}.4f}  {float(np.mean(va_losses)):>{w['loss']}.4f}  "
        f"{float(np.mean(gap_losses)):>+{w['gap']}.4f}  "
        f"{float(np.mean(tr_accs)):>{w['acc']}.4f}  {float(np.mean(va_accs)):>{w['acc']}.4f}  "
        f"{mean_gap_acc:>+{w['gap']}.4f}  "
        f"{float(np.mean(tr_f1s)):>{w['f1']}.4f}  {float(np.mean(va_f1s)):>{w['f1']}.4f}  "
        f"{float(np.mean(gap_f1s)):>+{w['gap']}.4f}  "
        f"{overfitting_severity(mean_gap_acc):>{w['sev']}}"
    )
    print(sep)
    print(
        f"\n  Overall overfitting verdict  →  {overfitting_severity(mean_gap_acc)}"
        f"  (mean ΔAcc={mean_gap_acc:+.4f}, mean ΔF1={float(np.mean(gap_f1s)):+.4f})\n"
    )


# ── CSV / printed results table ───────────────────────────────────────────────
_PER_EPOCH_FIELDS = [
    "fold", "epoch",
    "train_loss", "val_loss", "gap_loss",
    "train_acc",  "val_accuracy", "gap_acc",
    "train_f1",   "val_f1",       "gap_f1",
    "overfitting",
    "val_precision", "val_recall",
    "test_accuracy", "test_precision", "test_recall", "test_f1",
]


def _flat_row(fold: int, epoch: int, row: dict, m_test: dict) -> dict:
    m_val = row["val"]
    return {
        "fold": fold, "epoch": epoch,
        "train_loss": row["train_loss"], "val_loss": row["val_loss"], "gap_loss": row["gap_loss"],
        "train_acc":  row["train_acc"],  "val_accuracy": m_val["accuracy"], "gap_acc": row["gap_acc"],
        "train_f1":   row["train_f1"],   "val_f1": m_val["f1"],             "gap_f1": row["gap_f1"],
        "overfitting": row["overfitting"],
        "val_precision": m_val["precision"], "val_recall": m_val["recall"],
        "test_accuracy": m_test["accuracy"], "test_precision": m_test["precision"],
        "test_recall": m_test["recall"],     "test_f1": m_test["f1"],
    }


def save_and_print_results_tables(fold_reports: list[dict]) -> tuple[str, str]:
    os.makedirs(config.results_dir, exist_ok=True)
    prefix = config.results_file_prefix
    path_epoch = os.path.join(config.results_dir, f"{prefix}_per_epoch.csv")
    path_best  = os.path.join(config.results_dir, f"{prefix}_fold_best.csv")

    # Per-epoch CSV
    with open(path_epoch, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=_PER_EPOCH_FIELDS)
        w.writeheader()
        for fr in fold_reports:
            for row in fr["epoch_logs"]:
                w.writerow(_flat_row(row["fold"], row["epoch"], row, row["test"]))

    # Best-per-fold CSV
    keys = ["fold", "best_epoch"] + _PER_EPOCH_FIELDS[2:]
    best_rows: list[dict] = []
    for fr in fold_reports:
        b = fr["best_by_val_f1"]
        best_rows.append({
            "fold": fr["fold"], "best_epoch": b["epoch"],
            "train_loss": b["train_loss"], "val_loss": b["val_loss"], "gap_loss": b["gap_loss"],
            "train_acc":  b["train_acc"],  "val_accuracy": b["val"]["accuracy"], "gap_acc": b["gap_acc"],
            "train_f1":   b["train_f1"],   "val_f1": b["val"]["f1"],             "gap_f1": b["gap_f1"],
            "overfitting": b["overfitting"],
            "val_precision": b["val"]["precision"], "val_recall": b["val"]["recall"],
            "test_accuracy": b["test"]["accuracy"], "test_precision": b["test"]["precision"],
            "test_recall": b["test"]["recall"],     "test_f1": b["test"]["f1"],
        })

    mean_row: dict = {"fold": "MEAN", "best_epoch": ""}
    std_row:  dict = {"fold": "STD",  "best_epoch": ""}
    for k in keys[2:]:
        numeric = [r[k] for r in best_rows if isinstance(r[k], (int, float))]
        mean_row[k] = float(np.mean(numeric)) if numeric else ""
        std_row[k]  = float(np.std(numeric))  if numeric else ""

    with open(path_best, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in best_rows:
            w.writerow(row)
        w.writerow(mean_row)
        w.writerow(std_row)

    # ── Printed results table ─────────────────────────────────────────────────
    def fmt(x: object, w: int) -> str:
        s = f"{x:.4f}" if isinstance(x, float) else str(x)
        return s[:w].ljust(w)

    cw = {"fold": 6, "ep": 6, "m": 8}
    hdr = (
        f"{fmt('fold', cw['fold'])} {fmt('epoch', cw['ep'])} "
        f"{fmt('v_acc', cw['m'])} {fmt('v_P', cw['m'])} "
        f"{fmt('v_R',   cw['m'])} {fmt('v_F1', cw['m'])} "
        f"{fmt('t_acc', cw['m'])} {fmt('t_P', cw['m'])} "
        f"{fmt('t_R',   cw['m'])} {fmt('t_F1', cw['m'])}"
    )
    sep = "-" * len(hdr)
    lines = [
        "\n" + "=" * 100,
        "CLIP + FakeNewsNet RESULTS  (binary | pos_label=1=fake | WITH metadata)",
        "Best val-F1 epoch per fold   |  Acc / Precision / Recall / F1",
        "=" * 100,
        hdr, sep,
    ]
    for row in best_rows + [mean_row, std_row]:
        lines.append(
            f"{fmt(row['fold'], cw['fold'])} {fmt(row.get('best_epoch', ''), cw['ep'])} "
            f"{fmt(row['val_accuracy'],  cw['m'])} {fmt(row['val_precision'],  cw['m'])} "
            f"{fmt(row['val_recall'],    cw['m'])} {fmt(row['val_f1'],         cw['m'])} "
            f"{fmt(row['test_accuracy'], cw['m'])} {fmt(row['test_precision'], cw['m'])} "
            f"{fmt(row['test_recall'],   cw['m'])} {fmt(row['test_f1'],        cw['m'])}"
        )
    lines.append(sep)
    lines.append(f"Saved:\n  {os.path.abspath(path_epoch)}\n  {os.path.abspath(path_best)}")
    print("\n".join(lines))
    return path_epoch, path_best


# ── Training loop ─────────────────────────────────────────────────────────────
def train_fold(
    model: ClipFakeNewsDetector,
    train_loader: DataLoader,
    val_eval_loader: DataLoader,
    test_eval_loader: DataLoader,
    device: torch.device,
    fold_idx: int,
) -> dict:
    optimizer = torch.optim.AdamW(
        [
            {"params": model.backbone_params(), "lr": config.lr * config.backbone_lr_mult},
            {"params": model.head_params(),     "lr": config.lr},
        ],
        weight_decay=config.weight_decay,
    )
    t_total = len(train_loader) * config.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.06 * t_total),
        num_training_steps=t_total,
    )
    loss_fn = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)

    use_cuda = device.type == "cuda"
    autocast_ctx, scaler = build_amp_helpers(use_cuda)

    best_val_f1 = -1.0
    best_snapshot: dict | None = None
    epoch_logs: list[dict] = []

    for epoch in range(config.epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"Fold {fold_idx + 1} | Epoch {epoch + 1}/{config.epochs}")

        tr_loss_sum, tr_correct, tr_n = 0.0, 0, 0

        for step, batch in enumerate(pbar):
            labels = batch.pop("labels").to(device)
            n = labels.size(0)
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast_ctx():
                out = model(**batch, labels=labels, loss_fn=loss_fn)
                raw_loss = out["loss"]
                loss = raw_loss / config.grad_accumulation

            tr_loss_sum += raw_loss.detach().item() * n
            tr_n += n
            tr_correct += (out["logits"].detach().argmax(dim=-1) == labels).sum().item()

            if use_cuda:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % config.grad_accumulation == 0:
                if use_cuda:
                    scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), config.clip_grad_norm)
                if use_cuda:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f"{raw_loss.detach().item():.4f}")

        # ── Train metrics ─────────────────────────────────────────────────────
        train_loss = tr_loss_sum / max(tr_n, 1)
        train_acc  = tr_correct  / max(tr_n, 1)
        y_tr, p_tr, _ = collect_predictions_with_loss(
            model, train_loader, device, autocast_ctx, loss_fn
        )
        train_f1 = float(f1_score(y_tr, p_tr, average="binary", pos_label=1, zero_division=0))

        # ── Val / Test metrics ────────────────────────────────────────────────
        y_val, p_val, val_loss = collect_predictions_with_loss(
            model, val_eval_loader, device, autocast_ctx, loss_fn
        )
        y_te, p_te, _ = collect_predictions_with_loss(
            model, test_eval_loader, device, autocast_ctx, loss_fn
        )
        m_val  = binary_metrics(y_val, p_val)
        m_test = binary_metrics(y_te,  p_te)

        # ── Overfitting gaps ──────────────────────────────────────────────────
        gap_loss = val_loss  - train_loss
        gap_acc  = train_acc - m_val["accuracy"]
        gap_f1   = train_f1  - m_val["f1"]
        severity = overfitting_severity(gap_acc)

        epoch_log = {
            "fold": fold_idx + 1, "epoch": epoch + 1,
            "train_loss": train_loss, "val_loss": val_loss, "gap_loss": gap_loss,
            "train_acc":  train_acc,  "gap_acc": gap_acc,
            "train_f1":   train_f1,   "gap_f1": gap_f1,
            "overfitting": severity,
            "val": m_val, "test": m_test,
        }
        epoch_logs.append(epoch_log)

        print(
            f"\nFold {fold_idx + 1} | Epoch {epoch + 1}/{config.epochs}\n"
            f"  TRAIN → Loss={train_loss:.4f}  Acc={train_acc:.4f}  F1={train_f1:.4f}\n"
            f"  VAL   → Loss={val_loss:.4f}  Acc={m_val['accuracy']:.4f}  "
            f"P={m_val['precision']:.4f}  R={m_val['recall']:.4f}  F1={m_val['f1']:.4f}\n"
            f"  GAP   → ΔLoss={gap_loss:+.4f}  ΔAcc={gap_acc:+.4f}  ΔF1={gap_f1:+.4f}"
            f"  →  Overfitting: {severity}\n"
            f"  TEST  → Acc={m_test['accuracy']:.4f}  "
            f"P={m_test['precision']:.4f}  R={m_test['recall']:.4f}  F1={m_test['f1']:.4f}"
        )

        if m_val["f1"] > best_val_f1:
            best_val_f1 = m_val["f1"]
            best_snapshot = {
                "epoch": epoch + 1,
                "train_loss": train_loss, "val_loss": val_loss, "gap_loss": gap_loss,
                "train_acc":  train_acc,  "gap_acc": gap_acc,
                "train_f1":   train_f1,   "gap_f1": gap_f1,
                "overfitting": severity,
                "val": dict(m_val), "test": dict(m_test),
            }

    assert best_snapshot is not None
    print_fold_overfitting_table(fold_idx, epoch_logs)
    print(
        f"\n--- Fold {fold_idx + 1} best (val F1 @ epoch {best_snapshot['epoch']}) ---\n"
        f"  Train : Loss={best_snapshot['train_loss']:.4f}  Acc={best_snapshot['train_acc']:.4f}  "
        f"F1={best_snapshot['train_f1']:.4f}\n"
        f"  Val   : Loss={best_snapshot['val_loss']:.4f}  Acc={best_snapshot['val']['accuracy']:.4f}  "
        f"F1={best_snapshot['val']['f1']:.4f}\n"
        f"  GAP   : ΔLoss={best_snapshot['gap_loss']:+.4f}  ΔAcc={best_snapshot['gap_acc']:+.4f}  "
        f"ΔF1={best_snapshot['gap_f1']:+.4f}  → {best_snapshot['overfitting']}\n"
        f"  Test  : Acc={best_snapshot['test']['accuracy']:.4f}  "
        f"P={best_snapshot['test']['precision']:.4f}  R={best_snapshot['test']['recall']:.4f}  "
        f"F1={best_snapshot['test']['f1']:.4f}\n"
    )
    return {"fold": fold_idx + 1, "best_by_val_f1": best_snapshot, "epoch_logs": epoch_logs}


# ── Dataset loading ───────────────────────────────────────────────────────────
def _load_from_kaggle_hub() -> "datasets.Dataset":
    """
    Download / locate the dataset via kagglehub (works on Kaggle without
    external internet to huggingface.co) then build a HuggingFace Dataset
    from the local Parquet / CSV / JSON files found in the download path.
    """
    import glob
    import kagglehub
    from datasets import Dataset, load_dataset

    print(f"Attempting kagglehub download: {config.kaggle_dataset_slug} …")
    local_path = kagglehub.dataset_download(config.kaggle_dataset_slug)
    print(f"  kagglehub path: {local_path}")

    # Try Parquet first (most common for HF-mirrored datasets on Kaggle)
    parquet_files = sorted(glob.glob(os.path.join(local_path, "**", "*.parquet"), recursive=True))
    if parquet_files:
        print(f"  Found {len(parquet_files)} parquet file(s); loading …")
        ds_dict = load_dataset("parquet", data_files={"train": parquet_files})
        return ds_dict["train"]

    # Fallback: CSV
    csv_files = sorted(glob.glob(os.path.join(local_path, "**", "*.csv"), recursive=True))
    if csv_files:
        print(f"  Found {len(csv_files)} csv file(s); loading …")
        ds_dict = load_dataset("csv", data_files={"train": csv_files})
        return ds_dict["train"]

    # Fallback: JSON / JSONL
    json_files = sorted(glob.glob(os.path.join(local_path, "**", "*.json"), recursive=True))
    json_files += sorted(glob.glob(os.path.join(local_path, "**", "*.jsonl"), recursive=True))
    if json_files:
        print(f"  Found {len(json_files)} json file(s); loading …")
        ds_dict = load_dataset("json", data_files={"train": json_files})
        return ds_dict["train"]

    raise FileNotFoundError(
        f"No parquet/csv/json files found under kagglehub path: {local_path}"
    )


def load_hf_dataset():
    """
    Load FakeNewsNet with a two-stage strategy:
      1. Try HuggingFace Hub (works when internet is available).
      2. Fall back to kagglehub local download (works on Kaggle offline kernels).
    """
    from datasets import load_dataset

    # ── Stage 1: HuggingFace Hub ──────────────────────────────────────────
    if _HF_OFFLINE:
        print("Skipping HuggingFace Hub (HF_DATASETS_OFFLINE=1).")
        return _load_from_kaggle_hub()
    try:
        print("Trying HuggingFace Hub …")
        ds = load_dataset(config.dataset_name, split=config.dataset_split)
        print("  HuggingFace Hub load succeeded.")
        return ds
    except Exception as e_hf:
        print(f"  HuggingFace Hub failed ({type(e_hf).__name__}: {e_hf})")

    # Try without explicit split (some datasets expose DatasetDict)
    try:
        d = load_dataset(config.dataset_name)
        if hasattr(d, "__getitem__") and config.dataset_split in d:
            print("  HuggingFace Hub (DatasetDict) load succeeded.")
            return d[config.dataset_split]
        return next(iter(d.values()))
    except Exception as e_hf2:
        print(f"  HuggingFace Hub (DatasetDict) also failed ({type(e_hf2).__name__}: {e_hf2})")

    # ── Stage 2: kagglehub (Kaggle offline / internet-restricted env) ─────
    try:
        return _load_from_kaggle_hub()
    except Exception as e_kg:
        raise RuntimeError(
            f"Could not load dataset from HuggingFace Hub or kagglehub.\n"
            f"  HF error  : {e_hf}\n"
            f"  KH error  : {e_kg}\n"
            f"\nTo fix on Kaggle: add the dataset '{config.kaggle_dataset_slug}' "
            f"via Data → Add Data → Kaggle Datasets, then re-run."
        ) from e_kg


# ── Main ──────────────────────────────────────────────────────────────────────
def main() -> None:
    assert abs(config.train_frac + config.val_frac + config.test_frac - 1.0) < 1e-6

    device = resolve_training_device(force_cpu=config.force_cpu)
    print(f"Using device       : {device}")
    print(f"Model              : {config.model_name}")
    print(f"Dataset            : {config.dataset_name}  split={config.dataset_split}")
    print(f"Labels             : 0=real  1=fake")
    print(f"Metadata fields    : {config.metadata_fields}")
    print(f"Unfreeze last-N    : {config.unfreeze_last_n} CLIP transformer blocks per encoder")
    print(f"Backbone LR mult   : {config.backbone_lr_mult}  "
          f"(backbone lr = {config.lr * config.backbone_lr_mult:.2e})")
    print(f"Label smoothing    : {config.label_smoothing}")
    print(f"Grad clip norm     : {config.clip_grad_norm}")
    print(f"Dropout            : {config.dropout}")

    print("\nLoading FakeNewsNet …")
    ds = load_hf_dataset()
    print("Columns:", ds.column_names)

    n = min(config.max_samples, len(ds))
    ds = ds.select(range(n))
    print(f"Rows after max_samples cap: {len(ds)}")

    print("Loading CLIP processor …")
    processor = CLIPProcessor.from_pretrained(config.model_name)

    valid_idx = build_valid_indices(ds, len(ds))
    print(f"Valid samples (non-empty text + binary label): {len(valid_idx)}/{len(ds)}")

    y_all = np.array(
        [get_binary_label_fake_is_one(ds[i]) for i in valid_idx], dtype=np.int64
    )
    print("Class counts [real=0, fake=1]:", np.bincount(y_all, minlength=2).tolist())

    # ── 80 / 10 / 10 stratified split ────────────────────────────────────────
    pos = np.arange(len(valid_idx))
    pos_train, pos_temp, y_tr, y_tmp = train_test_split(
        pos, y_all,
        test_size=(config.val_frac + config.test_frac),
        stratify=y_all,
        random_state=SEED,
    )
    rel_test = config.test_frac / (config.val_frac + config.test_frac)
    pos_val, pos_test, _, _ = train_test_split(
        pos_temp, y_tmp,
        test_size=rel_test,
        stratify=y_tmp,
        random_state=SEED,
    )

    abs_train_pool = [valid_idx[int(i)] for i in pos_train]
    abs_val        = [valid_idx[int(i)] for i in pos_val]
    abs_test       = [valid_idx[int(i)] for i in pos_test]
    y_train_pool   = y_tr

    print(
        f"\nSplit sizes — "
        f"train pool : {len(abs_train_pool)} ({100 * len(abs_train_pool) / len(valid_idx):.1f}%)  "
        f"val : {len(abs_val)} ({100 * len(abs_val) / len(valid_idx):.1f}%)  "
        f"test : {len(abs_test)} ({100 * len(abs_test) / len(valid_idx):.1f}%)\n"
    )

    # ── Fixed val / test loaders ──────────────────────────────────────────────
    val_ds   = FakeNewsNetCLIPDataset(ds, processor, abs_val)
    test_ds  = FakeNewsNetCLIPDataset(ds, processor, abs_test)
    val_loader = DataLoader(
        val_ds, batch_size=config.batch_size, shuffle=False,
        num_workers=config.num_workers, collate_fn=collate_fn,
    )
    test_loader = DataLoader(
        test_ds, batch_size=config.batch_size, shuffle=False,
        num_workers=config.num_workers, collate_fn=collate_fn,
    )

    # ── 5-fold CV on train pool ───────────────────────────────────────────────
    skf = StratifiedKFold(n_splits=config.n_splits, shuffle=True, random_state=SEED)
    fold_reports: list[dict] = []

    for fold_idx, (tr_rel, _) in enumerate(skf.split(np.zeros(len(abs_train_pool)), y_train_pool)):
        print("\n" + "=" * 72)
        print(f"Fold {fold_idx + 1}/{config.n_splits}  (train subset from 80 % pool)")
        print("=" * 72)
        fold_train_abs = [abs_train_pool[i] for i in tr_rel]
        print(f"  This-fold train size: {len(fold_train_abs)}")

        train_ds = FakeNewsNetCLIPDataset(ds, processor, fold_train_abs)
        train_loader = DataLoader(
            train_ds, batch_size=config.batch_size, shuffle=True,
            num_workers=config.num_workers, collate_fn=collate_fn,
        )

        model = ClipFakeNewsDetector(config.num_classes).to(device)
        report = train_fold(model, train_loader, val_loader, test_loader, device, fold_idx)
        fold_reports.append(report)
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # ── Cross-fold summary ────────────────────────────────────────────────────
    print("\n" + "=" * 72)
    print("SUMMARY — best epoch per fold (by val F1, pos_label=fake=1)")
    print("=" * 72)
    for fr in fold_reports:
        b = fr["best_by_val_f1"]
        print(
            f"Fold {fr['fold']}: best_epoch={b['epoch']} | "
            f"VAL  Acc={b['val']['accuracy']:.4f} F1={b['val']['f1']:.4f} | "
            f"TEST Acc={b['test']['accuracy']:.4f} F1={b['test']['f1']:.4f}"
        )

    test_accs = [fr["best_by_val_f1"]["test"]["accuracy"]  for fr in fold_reports]
    test_prec = [fr["best_by_val_f1"]["test"]["precision"] for fr in fold_reports]
    test_rec  = [fr["best_by_val_f1"]["test"]["recall"]    for fr in fold_reports]
    test_f1s  = [fr["best_by_val_f1"]["test"]["f1"]        for fr in fold_reports]
    val_f1s   = [fr["best_by_val_f1"]["val"]["f1"]         for fr in fold_reports]

    print(
        f"\n{'='*72}\n"
        f"FINAL RESULTS  (mean ± std across {config.n_splits} folds, best-val-F1 epoch)\n"
        f"{'='*72}\n"
        f"  Val  F1         : {float(np.mean(val_f1s)):.4f} ± {float(np.std(val_f1s)):.4f}\n"
        f"  Test Accuracy   : {float(np.mean(test_accs)):.4f} ± {float(np.std(test_accs)):.4f}\n"
        f"  Test Precision  : {float(np.mean(test_prec)):.4f} ± {float(np.std(test_prec)):.4f}\n"
        f"  Test Recall     : {float(np.mean(test_rec)):.4f} ± {float(np.std(test_rec)):.4f}\n"
        f"  Test F1         : {float(np.mean(test_f1s)):.4f} ± {float(np.std(test_f1s)):.4f}\n"
        f"{'='*72}"
    )

    print_overfitting_report(fold_reports)
    save_and_print_results_tables(fold_reports)


if __name__ == "__main__":
    main()# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Using device       : cuda
Model              : openai/clip-vit-base-patch32
Dataset            : rickstello/FakeNewsNet  split=train
Labels             : 0=real  1=fake
Metadata           : NONE  (pure text + image only)
Unfreeze last-N    : 2 CLIP transformer blocks per encoder
Backbone LR mult   : 0.1  (backbone lr = 2.00e-06)
Label smoothing    : 0.1
Grad clip norm     : 1.0
Dropout            : 0.3

Loading FakeNewsNet …


README.md:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

FakeNewsNet.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/23196 [00:00<?, ? examples/s]

Columns: ['title', 'news_url', 'source_domain', 'tweet_num', 'real']
Rows after max_samples cap: 10000
Loading CLIP processor …


...


Fold 5 | Epoch 3/5
  TRAIN → Loss=0.4395  Acc=0.8473  F1=0.6237
  VAL   → Loss=0.4339  Acc=0.8430  P=0.8000  R=0.4647  F1=0.5879
  GAP   → ΔLoss=-0.0056  ΔAcc=+0.0043  ΔF1=+0.0358  →  Overfitting: None
  TEST  → Acc=0.8430  P=0.7862  R=0.4750  F1=0.5922


Fold 5 | Epoch 4/5:   0%|          | 0/400 [00:00<?, ?it/s]


Fold 5 | Epoch 4/5
  TRAIN → Loss=0.4296  Acc=0.8534  F1=0.6738
  VAL   → Loss=0.4285  Acc=0.8530  P=0.7670  R=0.5602  F1=0.6475
  GAP   → ΔLoss=-0.0012  ΔAcc=+0.0004  ΔF1=+0.0264  →  Overfitting: None
  TEST  → Acc=0.8460  P=0.7363  R=0.5583  F1=0.6351


Fold 5 | Epoch 5/5:   0%|          | 0/400 [00:00<?, ?it/s]


Fold 5 | Epoch 5/5
  TRAIN → Loss=0.4234  Acc=0.8578  F1=0.6724
  VAL   → Loss=0.4275  Acc=0.8510  P=0.7674  R=0.5477  F1=0.6392
  GAP   → ΔLoss=+0.0041  ΔAcc=+0.0068  ΔF1=+0.0332  →  Overfitting: None
  TEST  → Acc=0.8470  P=0.7514  R=0.5417  F1=0.6295

  ── Fold 5 Overfitting per Epoch ──
     Ep    TrLoss    VaLoss     ΔLoss    TrAcc    VaAcc      ΔAcc     TrF1     VaF1       ΔF1      Status
  -------------------------------------------------------------------------------------------------------
      1    0.5398    0.4632   -0.0766   0.7631   0.8230   -0.0599   0.4771   0.4779   -0.0008   Under-fit
      2    0.4566    0.4456   -0.0109   0.8300   0.8380   -0.0080   0.5510   0.5449   +0.0061   Under-fit
      3    0.4395    0.4339   -0.0056   0.8473   0.8430   +0.0043   0.6237   0.5879   +0.0358        None
      4    0.4296    0.4285   -0.0012   0.8534   0.8530   +0.0004   0.6738   0.6475   +0.0264        None
      5    0.4234    0.4275   +0.0041   0.8578   0.8510   +0.0068   0.6